In [1]:
# %load ./finetune.py
"""Finetune Qwen2.5-Coder-1.5B-Instruct on the nosh NL->bash dataset.

Train data = ./datasets/cleaned/000.csv ... 008.csv + a balanced subset of
nl2bash.csv. 009.csv and 010.csv are reserved for evaluation (eval.py).

Run:
    python finetune.py
    python finetune.py --epochs 5 --rank 64 --per-tool-cap 30
"""

import unsloth  # noqa: F401  must be imported before transformers/trl
from unsloth import FastLanguageModel

import argparse
import csv
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer
import random


ROOT = Path(".")
CLEANED_DIR = ROOT / "datasets" / "cleaned"
PROMPTS_DIR = ROOT / "prompts"

TEST_FILES = {"009.csv", "010.csv"}
NL2BASH_FILE = "nl2bash.csv"


def read_csv_robust(path: Path) -> pd.DataFrame:
    """Read a 2-column (input_text, bash_command) CSV tolerantly.

    Some rows have unquoted commas in input_text (e.g. ``cores 0,1``) so naive
    parsing splits them into 3+ fields. We use the csv stdlib (which is more
    forgiving than pandas) and re-join the leading fields into input_text.
    """
    rows = []
    with open(path, newline="") as f:
        reader = csv.reader(f)
        next(reader)  # skip header row
        for r in reader:
            if len(r) < 2:
                continue
            # Last field is bash_command; everything before it forms input_text
            input_text = ",".join(r[:-1])
            bash_command = r[-1]
            rows.append((input_text, bash_command))
    return pd.DataFrame(rows, columns=["input_text", "bash_command"])


def load_training_data(per_tool_cap: int, nl2bash_total_cap: int, seed: int):
    """Combine cleaned 000-008 with a balanced subset of nl2bash.

    nl2bash is dominated by `find` (~60%). We cap each tool (first whitespace
    token of bash_command) at `per_tool_cap`, then optionally trim to
    `nl2bash_total_cap` overall.
    """
    main_frames = []
    for path in sorted(CLEANED_DIR.glob("*.csv")):
        if path.name in TEST_FILES or path.name == NL2BASH_FILE:
            continue
        main_frames.append(read_csv_robust(path))
    main_df = pd.concat(main_frames, ignore_index=True)
    print(f"  cleaned 000-008: {len(main_df)} rows from {len(main_frames)} files")

    nl2bash = read_csv_robust(CLEANED_DIR / NL2BASH_FILE)
    tools = nl2bash["bash_command"].fillna("").str.strip().str.split().str[0]
    sampled = (
        nl2bash.groupby(tools, group_keys=False)
               .apply(lambda g: g.sample(min(len(g), per_tool_cap), random_state=seed))
               .reset_index(drop=True)
    )
    if len(sampled) > nl2bash_total_cap:
        sampled = sampled.sample(nl2bash_total_cap, random_state=seed).reset_index(drop=True)
    print(f"  nl2bash sampled: {len(sampled)} rows "
          f"(cap/tool={per_tool_cap}, total cap={nl2bash_total_cap})")

    full = pd.concat([main_df, sampled], ignore_index=True)
    full = (full.dropna(subset=["input_text", "bash_command"])
                .drop_duplicates(subset=["input_text", "bash_command"])
                .reset_index(drop=True))
    full = full.sample(frac=1).reset_index(drop=True)
    print(f"  combined train set: {len(full)} rows")
    return full


def load_prompt_assets():
    with open(PROMPTS_DIR / "nosh.md") as f:
        system_prompt = f.read().strip()
    with open(PROMPTS_DIR / "nosh_examples.json") as f:
        fewshots = json.load(f)
    return system_prompt, fewshots


class SamplePreviewCallback(TrainerCallback):
    """Generate predictions for a fixed set of prompts every N steps."""

    def __init__(self, model, tokenizer, rows, system_prompt, fewshots,
                 every_n_steps: int, sample_count: int, max_new_tokens: int = 128):
        self.model = model
        self.tokenizer = tokenizer
        self.rows = rows  # list of dicts with input_text + bash_command
        self.system_prompt = system_prompt
        self.fewshots = fewshots
        self.every_n_steps = every_n_steps
        self.max_new_tokens = max_new_tokens
        self.sample_count = sample_count

    def _generate(self, input_text: str) -> str:
        conversation = [{"role": "system", "content": self.system_prompt}]
        conversation.extend(self.fewshots)
        conversation.append({"role": "user", "content": input_text})
        text = self.tokenizer.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=0.1,
                pad_token_id=self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
                max_length=None
            )
        gen_ids = out[0][inputs.input_ids.shape[1]:]
        return self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    def _print_samples(self, step: int):
        was_training = self.model.training
        self.model.eval()
        try:
            print(f"\n=== sample previews @ step {step} ===")
            sampled_rows = random.sample(self.rows, self.sample_count)
            for row in sampled_rows:
                pred = self._generate(row["input_text"])
                ok = "OK " if pred == row["bash_command"] else "    "
                print(f"  {ok}input    : {row['input_text']}")
                print(f"      expected : {row['bash_command']}")
                print(f"      predicted: {pred}")
            print("=" * 40)
        finally:
            if was_training:
                self.model.train()

    def on_step_end(self, args, state, control, **kwargs):
        if self.every_n_steps <= 0:
            return
        if state.global_step > 0 and state.global_step % self.every_n_steps == 0:
            self._print_samples(state.global_step)

    def on_train_end(self, args, state, control, **kwargs):
        self._print_samples(state.global_step)


def make_sample(row, system_prompt, fewshots, tokenizer):
    conversation = [{"role": "system", "content": system_prompt}]
    conversation.extend(fewshots)
    conversation.append({"role": "user", "content": str(row["input_text"])})
    completion = [{"role": "assistant", "content": str(row["bash_command"])}]

    prompt_text = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=False
    )
    completion_text = tokenizer.apply_chat_template(
        completion, tokenize=False
    )
    return {"prompt": prompt_text, "completion": completion_text}



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", default="/home/paradox/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a")
    p.add_argument("--output-dir", default="./checkpoints/qwen2.5-coder-1.5b-nosh")
    p.add_argument("--epochs", type=float, default=3.0)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--grad-accum", type=int, default=2)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--warmup-steps", type=int, default=10)
    p.add_argument("--rank", type=int, default=32)
    p.add_argument("--alpha", type=int, default=32)
    p.add_argument("--dropout", type=float, default=0.0)
    p.add_argument("--max-seq-len", type=int, default=2048)
    p.add_argument("--per-tool-cap", type=int, default=50,
                   help="Max samples per first-token tool in nl2bash")
    p.add_argument("--nl2bash-total-cap", type=int, default=2000,
                   help="Hard cap on total nl2bash samples after per-tool capping")
    p.add_argument("--quantization", choices=["8bit", "4bit", "none"], default="8bit")
    p.add_argument("--seed", type=int, default=3407)
    p.add_argument("--save-merged", action="store_true",
                   help="Also save a merged-16bit copy for plug-in use with eval.py")
    p.add_argument("--no-padding-free", action="store_true",
                   help="Disable padding_free (needed on <17GB VRAM)")
    p.add_argument("--sample-every", type=int, default=50,
                   help="Print model predictions every N steps (0 disables)")
    p.add_argument("--sample-count", type=int, default=5,
                   help="Number of preview samples to generate each interval")
    p.add_argument("--sample-source", choices=["train", "test"], default="test",
                   help="Pull preview prompts from train data or held-out 009/010")
    return p.parse_args()

In [3]:
epochs=3
batch_size=16
grad_accum=2
lr=2e-4
warmup_steps=10
rank=32
alpha=32
dropout=0.0
max_seq_len=2048
per_tool_cap = 30
nl2bash_total_cap=1500
quantization="8bit"
seed=3407
save_merged=True
no_padding_free=False
sample_every=50
sample_count=5
sample_source="test"
modelpath = "/home/paradox/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a"
outputpath = "./checkpoints/qwen2.5-coder-1.5b-nosh"




In [4]:
                




print("Loading training data...")
train_df = load_training_data(
    per_tool_cap=per_tool_cap,
    nl2bash_total_cap=nl2bash_total_cap,
    seed=seed,
)

system_prompt, fewshots = load_prompt_assets()

print(f"Loading model: {modelpath} ({quantization})")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=modelpath,
    max_seq_length=max_seq_len,
    load_in_4bit=(quantization == "4bit"),
    load_in_8bit=(quantization == "8bit"),
    full_finetuning=False,
    token=""
)

model = FastLanguageModel.get_peft_model(
    model,
    r=rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=alpha,
    lora_dropout=dropout,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=seed,
    use_rslora=False,
    loftq_config=None,
)

print("Building prompts...")
samples = [make_sample(row, system_prompt, fewshots, tokenizer)
           for _, row in train_df.iterrows()]
train_dataset = Dataset.from_pandas(pd.DataFrame(samples))
print(f"  train_dataset: {len(train_dataset)} examples")



Loading training data...
  cleaned 000-008: 2121 rows from 9 files
  nl2bash sampled: 1500 rows (cap/tool=30, total cap=1500)
  combined train set: 3577 rows
Loading model: /home/paradox/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a (8bit)
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.51 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Building prompts...
  train_dataset: 3577 examples


In [6]:

preview_rows = []
if sample_every > 0:
    if sample_source == "test":
        test_frames = [read_csv_robust(CLEANED_DIR / name) for name in sorted(TEST_FILES)]
        pool = pd.concat(test_frames, ignore_index=True)
    else:
        pool = train_df
    preview_rows = (pool.to_dict(orient="records"))
    print(f"Sample previews every {sample_every} steps "
          f"({len(preview_rows)} rows from {sample_source})")

callbacks = []
if preview_rows:
    callbacks.append(SamplePreviewCallback(
        model=model,
        tokenizer=tokenizer,
        rows=preview_rows,
        system_prompt=system_prompt,
        fewshots=fewshots,
        every_n_steps=sample_every,
        sample_count=sample_count
    ))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=None,
    callbacks=callbacks,
    args=SFTConfig(
        output_dir=outputpath,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=warmup_steps,
        num_train_epochs=epochs,
        learning_rate=lr,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="cosine",
        seed=seed,
        report_to="none",
        padding_free=not no_padding_free,
        completion_only_loss=True,
        save_strategy="epoch",
        save_total_limit=2,
    ),
)



Sample previews every 50 steps (200 rows from test)


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=36):   0%|          | 0/3577 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free enabled, enabling faster training.


In [7]:

print("Training...")
trainer.train()



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,577 | Num Epochs = 3 | Total steps = 336
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.354412
10,1.553004
15,0.519236
20,0.304762
25,0.279534
30,0.326601
35,0.273038
40,0.287884
45,0.275737
50,0.276022


...

Unsloth: Restored added_tokens_decoder metadata in ./checkpoints/qwen2.5-coder-1.5b-nosh/checkpoint-224/tokenizer_config.json.



=== sample previews @ step 250 ===
      input    : terraform import an existing resource
      expected : terraform import YOUR_RESOURCE_TYPE YOUR_RESOURCE_ID
      predicted: terraform import resource_type.resource_name resource_id
      input    : check SSL/TLS protocols supported by a server
      expected : nmap --script ssl-enum-ciphers -p 443 YOUR_DOMAIN
      predicted: openssl s_client -connect example.com:443 2>/dev/null | openssl x509 -text -noout | grep "Supported signature algorithms"
      input    : run a Python script
      expected : python3 YOUR_SCRIPT
      predicted: python3 script.py
  OK input    : search command history for a keyword docker
      expected : history | grep docker
      predicted: history | grep docker
  OK input    : check for memory leaks in ./myapp
      expected : valgrind --leak-check=full ./myapp
      predicted: valgrind --leak-check=full ./myapp

=== sample previews @ step 300 ===
      input    : view logs for a specific service nginx
   

Unsloth: Restored added_tokens_decoder metadata in ./checkpoints/qwen2.5-coder-1.5b-nosh/checkpoint-336/tokenizer_config.json.



=== sample previews @ step 336 ===
  OK input    : run a Node.js script
      expected : node YOUR_SCRIPT
      predicted: node YOUR_SCRIPT
      input    : helm add a repository
      expected : helm repo add YOUR_REPO_NAME YOUR_REPO_URL
      predicted: helm repo add repo_name repo_url
      input    : view logs for a specific service nginx
      expected : journalctl -u nginx
      predicted: journalctl -u nginx -b
      input    : automate FTP file transfer
      expected : curl -T YOUR_FILE ftp://YOUR_SERVER --user YOUR_USER:YOUR_PASS
      predicted: ncftpput -u user password ftp.example.com /remote/path localfile.txt
      input    : change permissions recursively
      expected : chmod -R YOUR_PERMISSIONS YOUR_DIRECTORY
      predicted: chmod -R 755 directory/


TrainOutput(global_step=336, training_loss=0.24884649153266633, metrics={'train_runtime': 272.6771, 'train_samples_per_second': 39.354, 'train_steps_per_second': 1.232, 'total_flos': 1.2700637505819648e+16, 'train_loss': 0.24884649153266633, 'epoch': 3.0})

In [8]:
output_dir = outputpath

In [11]:

print(f"Saving LoRA adapter to {output_dir}")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)



Saving LoRA adapter to ./checkpoints/qwen2.5-coder-1.5b-nosh


('./checkpoints/qwen2.5-coder-1.5b-nosh/tokenizer_config.json',
 './checkpoints/qwen2.5-coder-1.5b-nosh/chat_template.jinja',
 './checkpoints/qwen2.5-coder-1.5b-nosh/tokenizer.json')

In [ ]:
/home/paradox/.cache/huggingface/hub/models--Qwen--Qwen2.5-Coder-1.5B-Instruct/snapshots/2e1fd397ee46e1388853d2af2c993145b0f1098a/config.json

In [ ]:
model.save_pretrained_merged()

In [25]:
model.config = model.config.to_dict()

In [26]:

if save_merged:
    merged_dir = f"{output_dir}-merged"
    print(f"Saving merged-16bit model to {merged_dir}")
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")

print("Done.")

Saving merged-16bit model to ./checkpoints/qwen2.5-coder-1.5b-nosh-merged


AttributeError: 'dict' object has no attribute '_name_or_path'